**Creación del CRM de retención**

In [1]:
import pandas as pd
import sqlite3

print("--- FASE 1: CREACIÓN DEL CRM DE RETENCIÓN ---")

# 1. Cargamos el archivo CSV con los resultados de riesgo calculados por el modelo predictivo
df_riesgo = pd.read_csv("../../datos/procesados/socios_prioritarios_ia.csv")

# 2. Ordenamos de mayor a menor riesgo y nos quedamos unicamente con los 50 casos mas urgentes
df_top_50 = df_riesgo.sort_values(by='probabilidad_churn', ascending=False).head(50)

ruta_db = '../../datos/crm_gimnasio.db'
conexion = sqlite3.connect(ruta_db)

# 3. Guardamos estos 50 socios en una tabla nueva dentro de nuestra base de datos local
df_top_50.to_sql('campana_retencion', conexion, if_exists='replace', index=False)

print(f"Base de datos SQLite creada en: {ruta_db}")
print(f"Se han cargado los {len(df_top_50)} socios más críticos en la tabla 'campana_retencion'.")

# 4. Hacemos una consulta rapida de lectura para verificar que la tabla se ha creado bien
df_verificacion = pd.read_sql("SELECT * FROM campana_retencion LIMIT 5", conexion)
print("\nMuestra de los 5 socios con más riesgo:")
display(df_verificacion)

conexion.close()

--- FASE 1: CREACIÓN DEL CRM DE RETENCIÓN ---
Base de datos SQLite creada en: ../../datos/crm_gimnasio.db
Se han cargado los 50 socios más críticos en la tabla 'campana_retencion'.

Muestra de los 5 socios con más riesgo:


,id_socio,probabilidad_churn,zona_proximidad,total_visitas,dias_sin_venir,edad,tipo_socio
0,4374.0,0.996239,1,191.0,186.0,40.0,AB. FAMILIAR
1,25538.0,0.991919,0,5.0,689.0,31.0,AB. FAMILIAR
2,23983.0,0.991425,2,83.0,216.0,17.0,AB. JOVE 14-17
3,27607.0,0.990425,0,51.0,322.0,25.0,AB. ATURAT
4,21984.0,0.988657,0,46.0,511.0,20.0,AB. FAMILIAR


**Prueba de Generación de Email Aleatorio**

In [7]:
import os
from dotenv import load_dotenv
from google import genai
import sqlite3
import pandas as pd

# 1. Autenticacion en la API de Google
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

# 2. Seleccion de un socio aleatorio
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')
socio_aleatorio = pd.read_sql("SELECT * FROM campana_retencion ORDER BY RANDOM() LIMIT 1", conexion).iloc[0]
conexion.close()

id_socio = int(socio_aleatorio['id_socio'])
dias_sin_venir = int(socio_aleatorio['dias_sin_venir'])
zona = int(socio_aleatorio['zona_proximidad'])
visitas = int(socio_aleatorio['total_visitas'])
edad = int(socio_aleatorio['edad'])
cuota = str(socio_aleatorio['tipo_socio'])

# 3. Reglas de negocio y preprocesamiento lógico
# Lógica de distancia
if zona == 0:
    texto_ubicacion = "Vive muy CERCA del gimnasio."
    regla_distancia = "Destaca la comodidad de tener el centro a un paso de casa."
else:
    texto_ubicacion = "Vive LEJOS del gimnasio."
    regla_distancia = "Empatiza con el desplazamiento y ofrece opciones virtuales."

# Lógica de franjas para las visitas
if visitas < 10:
    regla_visitas = "Anímale a retomar el impulso inicial. Recuérdale que los comienzos cuestan, pero lo importante es volver a dar el primer paso."
elif visitas <= 50:
    regla_visitas = "Menciónale que tenía un ritmo estupendo y anímale a recuperar esa buena rutina de entrenamiento que ya había conseguido."
else:
    regla_visitas = "Hazle saber que valoramos muchísimo su larga y constante trayectoria con nosotros. Anímale a no perder todo ese progreso acumulado."

# 4. Prompt Maestro
prompt = f"""
Actúa como el Director de Fidelización del centro deportivo Can Xaubet.
Escribe un email persuasivo y empático para recuperar al socio con ID {id_socio}.

CONTEXTO DEL SOCIO (INFORMACIÓN INTERNA):
- Días de ausencia: {dias_sin_venir} días.
- Ubicación: {texto_ubicacion}
- Edad: {edad} años.

REGLAS DE REDACCIÓN (IMPORTANTE):
1. SALUDO Y PRIVACIDAD: Usa "Estimado/a socio/a," (prohibido usar el ID). NUNCA menciones la cantidad exacta de días de ausencia.
2. TONO ADAPTADO A LA EDAD: Ajusta la madurez y energía del mensaje sabiendo que tiene {edad} años. 
3. APELA A SU HISTORIAL: {regla_visitas} ESTÁ TERMINANTEMENTE PROHIBIDO mencionar el número exacto de visitas que ha hecho.
4. PERSONALIZACIÓN POR DISTANCIA: {regla_distancia}
5. FLEXIBILIDAD DE CUOTA: Recuérdale que si sus circunstancias han cambiado, estamos a su disposición para adaptar su cuota o tarifa a sus necesidades actuales. NUNCA uses nombres técnicos de cuotas.
6. REGALO: Ofrécele una sesión gratuita de 30 min con un entrenador personal para reevaluar objetivos.
7. CIERRE: Despídete como "Director de Fidelización, Can Xaubet".
8. LÍMITE: Máximo 150 palabras.
"""

print(f"Extrayendo socio al azar de la base de datos...")
print(f"Generando comunicación para el socio {id_socio}")
print(f"Contexto de la IA: {dias_sin_venir} días sin venir, Zona {zona}, Edad {edad}, Visitas {visitas}, Cuota {cuota}\n")

# 5. Generación del correo
respuesta = client.models.generate_content(
    model='gemini-2.5-flash', 
    contents=prompt
)

print("EMAIL GENERADO ALEATORIAMENTE:")
print("-" * 50)
print(respuesta.text)
print("-" * 50)

Extrayendo socio al azar de la base de datos...
Generando comunicación para el socio 25538
Contexto de la IA: 689 días sin venir, Zona 0, Edad 31, Visitas 5, Cuota AB. FAMILIAR

EMAIL GENERADO ALEATORIAMENTE:
--------------------------------------------------
**Asunto:** ¡Can Xaubet te echa de menos!

Estimado/a socio/a,

Desde Can Xaubet queremos decirte que te echamos de menos y nos encantaría verte de nuevo. Sabemos que retomar la rutina a veces cuesta, pero recuerdas esa ilusión y energía inicial, ¿verdad? Lo importante es volver a dar ese primer paso y recuperar el ritmo.

Además, vivir tan cerca de Can Xaubet es una comodidad inigualable para tu día a día.

Si tus circunstancias han cambiado, no te preocupes. Estamos aquí para adaptar tu cuota o tarifa a tus necesidades actuales. Y para facilitarte el regreso, te ofrecemos una sesión gratuita de 30 minutos con un entrenador personal para reevaluar tus objetivos.

¡Esperamos tu visita!

Atentamente,

Director de Fidelización, Can 

**Código Original con API de Gemini (Desactivado hasta producción por límites de cuota gratuita)**

In [1]:
import os
import time
from dotenv import load_dotenv
from google import genai
import sqlite3
import pandas as pd

# 1. Configuracion de credenciales y conexion a la base de datos
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

conexion = sqlite3.connect('../../datos/crm_gimnasio.db')
df_campana = pd.read_sql("SELECT * FROM campana_retencion", conexion)

lista_emails = []

print(f"Iniciando generacion masiva para {len(df_campana)} socios...")
print("Este proceso tardara unos 4-5 minutos para no saturar la API...\n")

# 2. Bucle de generacion: iteramos sobre cada socio
for index, socio in df_campana.iterrows():
    id_socio = int(socio['id_socio'])
    dias_sin_venir = int(socio['dias_sin_venir'])
    zona = int(socio['zona_proximidad'])
    visitas = int(socio['total_visitas'])
    edad = int(socio['edad'])
    cuota = str(socio['tipo_socio'])

    # Lógica de distancia
    if zona == 0:
        texto_ubicacion = "Vive muy CERCA del gimnasio."
        regla_distancia = "Destaca la comodidad de tener el centro a un paso de casa."
    else:
        texto_ubicacion = "Vive LEJOS del gimnasio."
        regla_distancia = "Empatiza con el desplazamiento y ofrece opciones virtuales."

    # Lógica de franjas para las visitas
    if visitas < 10:
        regla_visitas = "Anímale a retomar el impulso inicial. Recuérdale que los comienzos cuestan, pero lo importante es volver a dar el primer paso."
    elif visitas <= 50:
        regla_visitas = "Menciónale que tenía un ritmo estupendo y anímale a recuperar esa buena rutina de entrenamiento que ya había conseguido."
    else:
        regla_visitas = "Hazle saber que valoramos muchísimo su larga y constante trayectoria con nosotros. Anímale a no perder todo ese progreso acumulado."

    # Prompt Maestro Definitivo
    prompt = f"""
    Actúa como el Director de Fidelización del centro deportivo Can Xaubet.
    Escribe un email persuasivo y empático para recuperar al socio con ID {id_socio}.

    CONTEXTO DEL SOCIO (INFORMACIÓN INTERNA):
    - Días de ausencia: {dias_sin_venir} días.
    - Ubicación: {texto_ubicacion}
    - Edad: {edad} años.

    REGLAS DE REDACCIÓN (IMPORTANTE):
    1. SALUDO Y PRIVACIDAD: Usa "Estimado/a socio/a," (prohibido usar el ID). NUNCA menciones la cantidad exacta de días de ausencia.
    2. TONO ADAPTADO A LA EDAD: Ajusta la madurez y energía del mensaje sabiendo que tiene {edad} años. 
    3. APELA A SU HISTORIAL: {regla_visitas} ESTÁ TERMINANTEMENTE PROHIBIDO mencionar el número exacto de visitas que ha hecho.
    4. PERSONALIZACIÓN POR DISTANCIA: {regla_distancia}
    5. FLEXIBILIDAD DE CUOTA: Recuérdale que si sus circunstancias han cambiado, estamos a su disposición para adaptar su cuota o tarifa a sus necesidades actuales. NUNCA uses nombres técnicos de cuotas.
    6. REGALO: Ofrécele una sesión gratuita de 30 min con un entrenador personal para reevaluar objetivos.
    7. CIERRE: Despídete como "Director de Fidelización, Can Xaubet".
    8. LÍMITE: Máximo 150 palabras.
    """

    # 3. Llamada a la API
    try:
        respuesta = client.models.generate_content(
            model='gemini-2.5-flash', 
            contents=prompt
        )
        texto_final = respuesta.text
        print(f"Email generado con exito para el socio {id_socio}")
        
    except Exception as e:
        texto_final = f"ERROR AL GENERAR: {e}"
        print(f"Fallo en el socio {id_socio} - Motivo: {e}")

    lista_emails.append(texto_final)
    time.sleep(5)

# 4. Actualizacion del CRM
df_campana['email_propuesto'] = lista_emails
df_campana.to_sql('campana_retencion', conexion, if_exists='replace', index=False)
conexion.close()

print("\nMision cumplida! Bucle finalizado y CRM actualizado con los 50 correos.")

Iniciando generacion masiva para 50 socios...
Este proceso tardara unos 4-5 minutos para no saturar la API...

Email generado con exito para el socio 4374
Email generado con exito para el socio 25538
Email generado con exito para el socio 23983
Email generado con exito para el socio 27607
Email generado con exito para el socio 21984
Email generado con exito para el socio 8991
Email generado con exito para el socio 23942
Email generado con exito para el socio 20800
Email generado con exito para el socio 25815
Email generado con exito para el socio 28113
Email generado con exito para el socio 25537
Email generado con exito para el socio 23348
Email generado con exito para el socio 14325
Email generado con exito para el socio 15794
Email generado con exito para el socio 26265
Email generado con exito para el socio 20369
Email generado con exito para el socio 24218
Email generado con exito para el socio 18470
Email generado con exito para el socio 12587
Email generado con exito para el soc

KeyboardInterrupt: 

**Comprobación Final de la Base de Datos**

In [1]:
import sqlite3
import pandas as pd

# 1. Establecemos la conexion directa con nuestra base de datos local SQLite
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')

# 2. Ejecutamos una consulta SQL seleccionando unicamente las columnas clave
# 3. Limitamos la extraccion a 5 socios para hacer una validacion rapida sin saturar la pantalla
df_resultado = pd.read_sql("SELECT id_socio, edad, total_visitas, zona_proximidad, email_propuesto FROM campana_retencion LIMIT 5", conexion)

conexion.close()

# 4. Renderizamos la tabla resultante para verificar visualmente la integracion de los textos
display(df_resultado)

,id_socio,edad,total_visitas,zona_proximidad,email_propuesto
0,4374.0,40.0,191.0,1,"Estimado/a socio/a, \nHace un tiempo que no no..."
1,25538.0,31.0,5.0,0,"Estimado/a socio/a, \nHace un tiempo que no no..."
2,23983.0,17.0,83.0,2,"Estimado/a socio/a, \nHace un tiempo que no no..."
3,27607.0,25.0,51.0,0,"Estimado/a socio/a, \nHace un tiempo que no no..."
4,21984.0,20.0,46.0,0,"Estimado/a socio/a, \nHace un tiempo que no no..."


**Generación Rápida de Correos (Modo MVP)**

In [2]:
import sqlite3
import pandas as pd

print("Iniciando rellenado rápido de base de datos (Modo MVP)")

# 1. Establecemos conexion con la base de datos para extraer los socios de la campana
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')
df_campana = pd.read_sql("SELECT * FROM campana_retencion", conexion)

# 2. Creamos plantillas de texto estaticas para simular el comportamiento de la IA
plantilla_cerca = """Estimado/a socio/a, 
Hace un tiempo que no nos vemos por Can Xaubet y notamos tu ausencia. 
Viviendo tan cerca, tienes la gran suerte de tener el centro a un paso de casa. Para animarte a retomar la rutina, queremos regalarte una sesión gratuita de 30 minutos con un entrenador personal para reevaluar tus objetivos. 
¡Con energía y salud! 

Atentamente, 
Director de Fidelización, 
Can Xaubet."""

plantilla_lejos = """Estimado/a socio/a, 
Hace un tiempo que no nos vemos por Can Xaubet. 
Sabemos que el desplazamiento puede ser un reto a veces, por lo que si lo necesitas, pregúntanos por nuestras opciones virtuales. Para animarte a volver con fuerza, te ofrecemos una sesión gratuita de 30 minutos con un entrenador personal. 
¡Con energía y salud! 

Atentamente, 
Director de Fidelización, 
Can Xaubet."""

# 3. Iteramos sobre los socios y asignamos el modelo de correo segun su distancia al centro
lista_emails = []
for index, socio in df_campana.iterrows():
    zona = int(socio['zona_proximidad'])
    if zona == 0:
        lista_emails.append(plantilla_cerca)
    else:
        lista_emails.append(plantilla_lejos)

# 4. Guardamos los correos asignados como una nueva columna y actualizamos la base de datos
df_campana['email_propuesto'] = lista_emails
df_campana.to_sql('campana_retencion', conexion, if_exists='replace', index=False)
conexion.close()

print("Base de datos actualizada con los 50 correos simulados.")

Iniciando rellenado rápido de base de datos (Modo MVP)
Base de datos actualizada con los 50 correos simulados.


In [ ]:
import sqlite3
import pandas as pd

# 1. Conectamos a la base de datos donde la IA guardó los correos
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')

# 2. Leemos la tabla completa de la campaña
df_final_powerbi = pd.read_sql("SELECT * FROM campana_retencion", conexion)

# 3. La exportamos a la carpeta de procesados para Power BI
ruta_pbi = '../../datos/procesados/campana_retencion_final.csv'
df_final_powerbi.to_csv(ruta_pbi, index=False)

conexion.close()

print(f"El archivo para Power BI se ha guardado en:\n{ruta_pbi}")

¡Hecho! El archivo para Power BI se ha guardado en:
../../datos/procesados/campana_retencion_final.csv
